# 04 — Readmission Analysis

Detects 30-day hospital readmissions and identifies which diagnoses, DRGs, and beneficiary groups have elevated readmission risk.

A readmission occurs when a patient is discharged and readmitted within 30 days.

Requires `cms_data.db` from `01_setup.ipynb`.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

---
## 1. Flag 30-day readmissions

In [ ]:
# Load inpatient claims, sort by patient and date
claims = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  CLM_ID,
  CLM_ADMSN_DT,
  NCH_BENE_DSCHRG_DT,
  CLM_DRG_CD,
  ICD9_DGNS_CD_1,
  CLM_PMT_AMT
FROM inpatient_claims
WHERE CLM_ADMSN_DT IS NOT NULL
ORDER BY DESYNPUF_ID, CLM_ADMSN_DT
""", conn)

# Convert date strings to datetime
claims['CLM_ADMSN_DT'] = pd.to_datetime(claims['CLM_ADMSN_DT'], format='%Y%m%d')
claims['NCH_BENE_DSCHRG_DT'] = pd.to_datetime(claims['NCH_BENE_DSCHRG_DT'], format='%Y%m%d')

# For each patient, calculate days to next admission
claims['next_admission_date'] = claims.groupby('DESYNPUF_ID')['CLM_ADMSN_DT'].shift(-1)
claims['days_to_readmission'] = (claims['next_admission_date'] - claims['NCH_BENE_DSCHRG_DT']).dt.days

# Flag readmission if next admission within 30 days
claims['readmitted_30d'] = (claims['days_to_readmission'] > 0) & (claims['days_to_readmission'] <= 30)

print(f"Total inpatient stays: {len(claims):,}")
print(f"Readmitted within 30 days: {claims['readmitted_30d'].sum():,} ({claims['readmitted_30d'].mean()*100:.1f}%)")
print(f"Not readmitted (or >30 days): {(~claims['readmitted_30d']).sum():,}")

---
## 2. Readmission rates by DRG

In [ ]:
drg_readmit = claims.groupby('CLM_DRG_CD').agg(
    num_admissions=('CLM_ID', 'count'),
    readmissions=('readmitted_30d', 'sum')
).reset_index()

drg_readmit['readmission_rate'] = (drg_readmit['readmissions'] / drg_readmit['num_admissions'] * 100).round(1)
drg_readmit = drg_readmit.sort_values('readmission_rate', ascending=False).reset_index(drop=True)

print("DRGs with Highest 30-Day Readmission Rates (top 20)")
print("="*80)
display(drg_readmit.head(20))
drg_readmit.to_csv("../data/exports/readmission_rates_by_drg.csv", index=False)

---
## 3. Readmission rates by primary diagnosis

In [ ]:
diag_readmit = claims[claims['ICD9_DGNS_CD_1'].notna()].groupby('ICD9_DGNS_CD_1').agg(
    num_admissions=('CLM_ID', 'count'),
    readmissions=('readmitted_30d', 'sum')
).reset_index()

diag_readmit['readmission_rate'] = (diag_readmit['readmissions'] / diag_readmit['num_admissions'] * 100).round(1)
diag_readmit = diag_readmit.sort_values('readmission_rate', ascending=False).reset_index(drop=True)

print("Primary Diagnoses with Highest 30-Day Readmission Rates (top 20)")
print("="*80)
display(diag_readmit.head(20))
diag_readmit.to_csv("../data/exports/readmission_rates_by_diagnosis.csv", index=False)

---
## 4. Frequent readmitters (beneficiaries with multiple readmissions)

In [ ]:
# Count readmissions per beneficiary
readmitter_profile = claims.groupby('DESYNPUF_ID').agg(
    total_admits=('CLM_ID', 'count'),
    num_readmitted=('readmitted_30d', 'sum'),
    avg_cost_per_admit=('CLM_PMT_AMT', 'mean')
).reset_index()

readmitter_profile['readmit_rate'] = (readmitter_profile['num_readmitted'] / readmitter_profile['total_admits'] * 100).round(1)

# Identify frequent readmitters (2+ readmissions)
frequent_readmitters = readmitter_profile[readmitter_profile['num_readmitted'] >= 2].sort_values('num_readmitted', ascending=False)

print(f"
Beneficiaries with ≥2 readmissions: {len(frequent_readmitters):,}")
print(f"Total readmissions from this group: {frequent_readmitters['num_readmitted'].sum():,}")
print(f"
Avg characteristics:")
print(f"  Total admits: {frequent_readmitters['total_admits'].mean():.1f}")
print(f"  Readmission rate: {frequent_readmitters['readmit_rate'].mean():.1f}%")
print(f"  Avg cost per admit: ${frequent_readmitters['avg_cost_per_admit'].mean():,.0f}")

print(f"
Top 20 frequent readmitters:")
display(frequent_readmitters.head(20))

---
## 5. Readmission risk by chronic condition status

In [ ]:
# Join claims with beneficiary chronic condition data
bene_chronic = pd.read_sql("""
SELECT DISTINCT
  DESYNPUF_ID,
  SP_CHF,
  SP_COPD,
  SP_DIABETES,
  SP_ISCHMCHT,
  SP_DEPRESSN,
  SP_CHRNKIDN
FROM beneficiary_summary
""", conn)

claims_with_chronic = claims.merge(bene_chronic, on='DESYNPUF_ID', how='left')

chronic_readmit = pd.DataFrame()
for condition in ['SP_CHF', 'SP_COPD', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_DEPRESSN', 'SP_CHRNKIDN']:
    with_cond = claims_with_chronic[claims_with_chronic[condition] == 1]
    without_cond = claims_with_chronic[claims_with_chronic[condition] == 2]
    
    row = {
        'condition': condition,
        'readmit_rate_with_condition': (with_cond['readmitted_30d'].sum() / len(with_cond) * 100).round(1) if len(with_cond) > 0 else None,
        'readmit_rate_without_condition': (without_cond['readmitted_30d'].sum() / len(without_cond) * 100).round(1) if len(without_cond) > 0 else None,
    }
    chronic_readmit = pd.concat([chronic_readmit, pd.DataFrame([row])], ignore_index=True)

chronic_readmit['risk_increase'] = (chronic_readmit['readmit_rate_with_condition'] - chronic_readmit['readmit_rate_without_condition']).round(1)

print("30-Day Readmission Rates: With vs Without Chronic Conditions")
print("="*90)
display(chronic_readmit)
chronic_readmit.to_csv("../data/exports/readmission_by_chronic_condition.csv", index=False)

conn.close()
print("
Readmission analysis complete. Outputs saved to data/exports/")